# Tutorial

In [1]:
import pandas as pd
import pint
ureg = pint.get_application_registry()

In [7]:
from aircraftdetective.processing.usdot import process_data_usdot_t2
from aircraftdetective.processing.acftdb import _read_engine_database
from aircraftdetective.processing.literature import (
    process_data_weinold_database,
    process_data_babikian_figures
)

from aircraftdetective.calculations.engines import (
    determine_takeoff_to_cruise_tsfc_ratio,
    scale_engine_data_from_icao_emissions_database
)
from aircraftdetective.calculations.engines import (
    calculate_air_mass_flow_rate,
    calculate_engine_efficiencies
)
from aircraftdetective.calculations.aerodynamics import (
    compute_lift_to_drag_ratio,
    compute_aspect_ratio
)

from aircraftdetective.utility.tabular import (
    left_merge_wildcard,
    export_typed_dataframe_to_excel
)

Aircraft specifications are provided in an Excel file hosted at the Zenodo repository ["Aircraft Detective" Dataset](https://doi.org/10.5281/zenodo.14382100).  
They can be loaded using the [`aircraftdetective.processing.literature.process_data_weinold_database`](https://aircraftdetective.readthedocs.io/en/latest/api/processing/literature/#aircraftdetective.processing.literature.process_data_weinold_database) function.

In [8]:
df_acft = process_data_weinold_database()

In [9]:
df_acft.head(5)

,Manufacturer,Aircraft Designation,Aircraft Designation (Literature),Aircraft Designation (aircraft-database.com),Aircraft Designation (US DOT Schedule T2),Type,YOI,Engine Designation (ICAO),Engine Designation (aircraft-database.com),Design Range,...,MZFW,Number of Engines,Sources (Weights),Comments,Payload/Range: Range at Point B,Payload/Range: Range at Point C,Payload/Range: MZFW at Point B,Payload/Range: MZFW at Point C,Payload/Range: MTOW,Source (Payload/Range)
0,Airbus,A220-100,NaN,A220-100,A200-100 BD-500-1A10,Narrow,2016,PW1519G,PW1519G,6700.0,...,50349.0,2.0,https://aircraft.airbus.com/sites/g/files/jlcb...,NaN,3598.3749284019223,7158.606627935522,50350.0026357575,43169.9977927272,60781.0,https://aircraft.airbus.com/sites/g/files/jlcb...
1,Airbus,A220-300,NaN,A220-300,A220-300 BD-500-1A11,Narrow,2016,PW1521G,PW1521G,6300.0,...,55792.0,2.0,https://aircraft.airbus.com/sites/g/files/jlcb...,NaN,3681.3845400518944,6230.448593959429,55791.8615,49986.0186311349,67585.0,https://aircraft.airbus.com/sites/g/files/jlcb...
2,Airbus,A300-600,A300-600,A300 B4-600,Airbus Industrie A300-600/R/CF/RCF,Wide,1984,CF6-80C2A5,CF6-80C2A5,6852.0,...,130000.0,2.0,[JAWA:07/08],NaN,3621.6888888888893,5700.044444444445,129255.84717830301,115645.08717830303,165035.00191999998,https://www.airbus.com/sites/g/files/jlcbta136...
3,Airbus,A300-600,A300-600,A300 B4-601,Airbus Industrie A300-600/R/CF/RCF,Wide,1986,PW4158,PW4158,6852.0,...,130000.0,2.0,[JAWA:07/08],NaN,3621.6888888888893,5700.044444444445,129255.84717830301,115645.08717830303,165035.00191999998,https://www.airbus.com/sites/g/files/jlcbta136...
4,Airbus,A310-200,NaN,A310-200,Airbus Industrie A310-200C/F,Wide,1983,JT9D-7R4D*,JT9D-7R4D*,6500.0,...,nan,2.0,[JAWA:07/08],NaN,nan,nan,nan,nan,nan,https://www.airbus.com/sites/g/files/jlcbta136...


Engine specifications are provided in an Excel file hosted at the Zenodo repository ["Aircraft Detective" Dataset](https://doi.org/10.5281/zenodo.14382100).

In [10]:
dict_tsfc_scaling = determine_takeoff_to_cruise_tsfc_ratio(degree=2, plot=True)
df_engines_scaled = scale_engine_data_from_icao_emissions_database(
    scaling_polynomial=dict_tsfc_scaling['TSFC (cruise)'],
)
df_engines_scaled.drop(columns=['Final Test Date'], inplace=True)

In [11]:
df_engines = _read_engine_database()

In [12]:
df_merged = left_merge_wildcard(
    df_left=df_acft,
    df_right=df_engines_scaled,
    left_on='Engine Designation (ICAO)',
    right_on='Engine Identification',
)
df_merged = left_merge_wildcard(
    df_left=df_merged,
    df_right=df_engines,
    left_on='Engine Designation (aircraft-database.com)',
    right_on='Engine Designation',
)

In [14]:
df_merged.head(4)

,Manufacturer,Aircraft Designation,Aircraft Designation (Literature),Aircraft Designation (aircraft-database.com),Aircraft Designation (US DOT Schedule T2),Type,YOI,Engine Designation (ICAO),Engine Designation (aircraft-database.com),Design Range,...,TSFC (takeoff),TSFC (cruise),Bypass Ratio,Overall Pressure Ratio,Dry Weight,Fan Diameter,Max. Continuous Thrust,_id_engine,Engine Family,Engine Manufacturer
0,Airbus,A220-100,NaN,A220-100,A200-100 BD-500-1A10,Narrow,2016,PW1519G,PW1519G,6700.0,...,7.048658481127786,13.44781176238713,nan,nan,2177.0,1.85,83.12,1ec8d740-0103-60f2-860d-69edf93e810e,turbofan,Pratt & Whitney
1,Airbus,A220-300,NaN,A220-300,A220-300 BD-500-1A11,Narrow,2016,PW1521G,PW1521G,6300.0,...,7.060990585345886,13.465996936432283,nan,nan,2177.0,1.85,92.35,1ec8d742-7536-6fda-860d-9318256fabb1,turbofan,Pratt & Whitney
2,Airbus,A300-600,A300-600,A300 B4-600,Airbus Industrie A300-600/R/CF/RCF,Wide,1984,CF6-80C2A5,CF6-80C2A5,6852.0,...,9.654088834576154,16.912337735297225,5.1,nan,4300.0,2.36,250.03,1ec89968-74fb-670e-b99a-cf6db5bd217d,turbofan,General Electric
3,Airbus,A300-600,A300-600,A300 B4-601,Airbus Industrie A300-600/R/CF/RCF,Wide,1986,PW4158,PW4158,6852.0,...,9.616279069767442,16.867485204335395,4.8,30.7,4273.0,2.39,220.54,1ec8c156-ca49-61aa-860d-a11b73be09ea,turbofan,Pratt & Whitney


In [15]:
df_merged = calculate_air_mass_flow_rate(df_merged)

In [16]:
df_t2 = process_data_usdot_t2()
df_t2 = df_t2.groupby('Aircraft Designation (US DOT Schedule T2)').mean()
df_with_dot = pd.merge(
    how='left',
    left=df_merged,
    right=df_t2,
    left_on='Aircraft Designation (US DOT Schedule T2)',
    right_on='Aircraft Designation (US DOT Schedule T2)'
)

In [17]:
df_with_dot = calculate_engine_efficiencies(df=df_with_dot)
df_with_dot = compute_aspect_ratio(df_with_dot)
df_with_dot = compute_lift_to_drag_ratio(df_with_dot, beta=0.04)

In [79]:
df_with_dot

,Manufacturer,Aircraft Designation,Aircraft Designation (Literature),Aircraft Designation (aircraft-database.com),Aircraft Designation (US DOT Schedule T2),Type,YOI,Engine Designation (ICAO),Engine Designation (aircraft-database.com),Design Range,...,Fuel/Revenue Seat Distance,Fuel Flow,Energy Use (per ASK),Airborne Efficiency,SLF,Engine Efficiency,Propulsive Efficiency,Thermal Efficiency,Aspect Ratio,L/D
0,Airbus,A220-100,NaN,A220-100,A200-100 BD-500-1A10,Narrow,2016,PW1519G,PW1519G,6700.0,...,0.019219012506973217,672.7058856593372,497675.6290018066,0.8278189758393693,0.7810071307439735,1.428333042570806e-06,0.8073414771820275,1.7691808025970729e-06,10.970703472840606,16.662689537570333
1,Airbus,A220-300,NaN,A220-300,A220-300 BD-500-1A11,Narrow,2016,PW1521G,PW1521G,6300.0,...,0.016912406439006062,840.4810493380713,464303.5398023028,0.8150626646059727,0.804697847717592,1.426404148253051e-06,0.7705664741746658,1.8511110930187252e-06,10.970703472840606,14.299599889133317
2,Airbus,A300-600,A300-600,A300 B4-600,Airbus Industrie A300-600/R/CF/RCF,Wide,1984,CF6-80C2A5,CF6-80C2A5,6852.0,...,0.02339057742031206,1967.2451931194253,580139.0168358667,0.8592071245393107,0.7205480386010795,1.1412162119658897e-06,0.7475432276078617,1.5266223675355625e-06,7.729726538461538,16.913520025725198
3,Airbus,A300-600,A300-600,A300 B4-601,Airbus Industrie A300-600/R/CF/RCF,Wide,1986,PW4158,PW4158,6852.0,...,0.02339057742031206,1967.2451931194253,580139.0168358667,0.8592071245393107,0.7205480386010795,1.1442508335980576e-06,0.751785921057727,1.5220434455438474e-06,7.729726538461538,16.868664359258453
4,Airbus,A310-200,NaN,A310-200,Airbus Industrie A310-200C/F,Wide,1983,JT9D-7R4D*,JT9D-7R4D*,6500.0,...,0.03467393473675327,1696.4448412450565,716685.7971126562,0.8886528414373854,0.6141077924712185,1.1668621911251597e-06,0.7825336116487677,1.4911336379106154e-06,7.729726538461538,nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124,McDonnell Douglas,DC-10-10,DC10-10,DC-10,McDonnell Douglas DC-10-10,Wide,1971,NaN,NaN,nan,...,0.030647638270900737,2725.221503378348,696965.4557115521,0.8697766384618486,0.6837898267755348,nan,nan,nan,nan,nan
125,McDonnell Douglas,DC-10-30,DC10-30,DC-10,McDonnell Douglas DC-10-30,Wide,1972,NaN,NaN,nan,...,0.02900991425246431,2944.6941236017815,740880.7142421949,0.9200260927951432,0.7622711729058671,nan,nan,nan,nan,nan
126,McDonnell Douglas,DC-10-40,DC10-40,DC-10,McDonnell Douglas DC-10-40,Wide,1973,NaN,NaN,nan,...,0.028344709197408655,2894.4190792012428,704852.0861302295,0.9006058031346184,0.7303027448385714,nan,nan,nan,nan,nan
127,Lockheed,L-1011-100,L1011-1/100/200,L-1011 TriStar,Lockheed L100-10 Hercules,Wide,1975,NaN,NaN,nan,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan


In [80]:
df_with_dot['Energy Use (per ASK)'] = df_with_dot['Energy Use (per ASK)'].pint.to('MJ/km')

In [81]:
df_with_dot['Energy Use (per ASK)']

0      1.1706056571081447
1      1.0921096365479033
2      1.3645715712045796
3      1.3645715712045796
4       1.685749511487712
              ...        
124     1.639364392629194
125    1.7426594850018018
126    1.6579148975075262
127                   nan
128                   nan
Name: Energy Use (per ASK), Length: 129, dtype: pint[megajoule / kilometer][Float64]

In [82]:
from aircraftdetective.calculations.weight import calculate_weight_metrics

In [83]:
df_with_dot = calculate_weight_metrics(df_with_dot)

In [84]:
from aircraftdetective.utility.tabular import update_column_data

In [85]:
df_literature = process_data_weinold_database(sheet_name='Literature Data')

In [86]:
df_literature.dtypes

Aircraft Designation (Literature)                                       object
L/D                                               pint[dimensionless][float64]
Source (L/D)                                                            object
Energy Use (per ASK)                      pint[megajoule / kilometer][float64]
Source (Energy Use)                                                     object
TSFC (cruise)                        pint[gram / kilonewton / second][float64]
Source (TSFC (Cruise))                                                  object
OEW/MTOW                                          pint[dimensionless][float64]
Source (OEW/MTOW)                                                       object
dtype: object

In [87]:
df_literature

,Aircraft Designation (Literature),L/D,Source (L/D),Energy Use (per ASK),Source (Energy Use),TSFC (cruise),Source (TSFC (Cruise)),OEW/MTOW,Source (OEW/MTOW)
0,A300-600,14.6026,https://doi.org/10.5281/zenodo.14560914,1.3176,https://doi.org/10.5281/zenodo.14560914,16.8925,https://doi.org/10.5281/zenodo.14560914,0.52078,https://doi.org/10.5281/zenodo.14560914
1,A310-300,16.499,https://doi.org/10.5281/zenodo.14560914,1.4459,https://doi.org/10.5281/zenodo.14560914,16.8925,https://doi.org/10.5281/zenodo.14560914,0.50967,https://doi.org/10.5281/zenodo.14560914
2,A320-100/200,15.9818,https://doi.org/10.5281/zenodo.14560914,1.152,https://doi.org/10.5281/zenodo.14560914,16.8925,https://doi.org/10.5281/zenodo.14560914,0.55411,https://doi.org/10.5281/zenodo.14560914
3,A380-800,nan,NaN,1.375286583998318,Rohrer et al.,nan,NaN,nan,NaN
4,B707-300,15.8095,https://doi.org/10.5281/zenodo.14560914,3.0372,https://doi.org/10.5281/zenodo.14560914,28.1765,https://doi.org/10.5281/zenodo.14560914,28.1765,https://doi.org/10.5281/zenodo.14560914
5,B720-000,15.8095,https://doi.org/10.5281/zenodo.14560914,3.0372,https://doi.org/10.5281/zenodo.14560914,nan,NaN,27.0278,https://doi.org/10.5281/zenodo.14560914
6,B720-000,nan,NaN,3.2365,https://doi.org/10.5281/zenodo.14560914,nan,NaN,nan,NaN
7,B727-200/231A,13.798,https://doi.org/10.5281/zenodo.14560914,2.0912,https://doi.org/10.5281/zenodo.14560914,23.244,https://doi.org/10.5281/zenodo.14560914,0.57158,https://doi.org/10.5281/zenodo.14560914
8,B737-100/200,13.9129,https://doi.org/10.5281/zenodo.14560914,1.9257,https://doi.org/10.5281/zenodo.14560914,22.771,https://doi.org/10.5281/zenodo.14560914,0.51984,https://doi.org/10.5281/zenodo.14560914
9,B737-300,14.0854,https://doi.org/10.5281/zenodo.14560914,1.3885,https://doi.org/10.5281/zenodo.14560914,20.8791,https://doi.org/10.5281/zenodo.14560914,0.54777,https://doi.org/10.5281/zenodo.14560914


In [100]:
dftest = update_column_data(
    df_main=df_with_dot,
    df_other=df_literature,
    merge_column='Aircraft Designation (Literature)',
    list_columns=['L/D', 'Energy Use (per ASK)', 'TSFC (cruise)', 'OEW/MTOW']
)

In [ ]:
dftest

In [89]:
df_with_dot.columns

Index(['Manufacturer', 'Aircraft Designation',
       'Aircraft Designation (Literature)',
       'Aircraft Designation (aircraft-database.com)',
       'Aircraft Designation (US DOT Schedule T2)', 'Type', 'YOI',
       'Engine Designation (ICAO)',
       'Engine Designation (aircraft-database.com)', 'Design Range',
       'Sources (Design Range)', 'Cruise Speed', 'Sources (Cruise Speed)',
       'Wingspan', 'Wing Area', 'Sources (Wing Dimensions)', 'Pax Exit Limit',
       'Average Pax', 'Sources (Pax)', 'OEW', 'MTOW', 'MZFW',
       'Number of Engines', 'Sources (Weights)', 'Comments',
       'Payload/Range: Range at Point B', 'Payload/Range: Range at Point C',
       'Payload/Range: ZFW at Point B', 'Payload/Range: ZFW at Point C',
       'Payload/Range: MTOW', 'Source (Payload/Range)', 'All Data hamonized?',
       'Fuel Flow (takeoff)', 'Fuel Flow (climbout)', 'Fuel Flow (approach)',
       'Fuel Flow (idle)', 'B/P Ratio', 'Pressure Ratio', 'Rated Thrust',
       'TSFC (takeoff)',

In [90]:
from aircraftdetective.calculations.decomposition import compute_efficiency_improvement_metrics

In [91]:
from aircraftdetective.utility.statistics import _compute_polynomials_from_dataframe

In [92]:
dftest = df_with_dot[['Aircraft Designation', 'YOI', 'Energy Use (per ASK)']]

In [93]:
df_t2['Energy Use (per ASK)'] = df_t2['Energy Use (per ASK)'].pint.to('MJ/km')

In [94]:
df_t2[df_t2.index == 'Airbus Industrie A310-200C/F']

,Year,Fuel/Available Seat Distance,Fuel/Revenue Seat Distance,Fuel Flow,Energy Use (per ASK),Airborne Efficiency,SLF
Aircraft Designation (US DOT Schedule T2),,,,,,,
Airbus Industrie A310-200C/F,1992.25,0.02065376936924081,0.03467393473675327,1696.4448412450565,1.685749511487712,0.8886528414373854,0.6141077924712185


In [68]:
dftest

,Aircraft Designation,YOI,Energy Use (per ASK)
0,A220-100,2016,1.1706056571081447
1,A220-300,2016,1.0921096365479033
2,A300-600,1984,1.3645715712045796
3,A300-600,1986,1.3645715712045796
4,A310-200,1983,74.39691724745963
...,...,...,...
124,DC-10-10,1971,1.639364392629194
125,DC-10-30,1972,1.7426594850018018
126,DC-10-40,1973,1.8298391088003219
127,L-1011-100,1975,nan


In [95]:
_compute_polynomials_from_dataframe(
    df=df_with_dot,
    col_name_x='YOI',
    list_col_names_y=['Energy Use (per ASK)'],
    degree=2,
    plot=True
)

{'Energy Use (per ASK)': Polynomial([ 1.56486511, -0.40996086, -0.11409516], domain=[1964., 2018.], window=[-1.,  1.], symbol='x'),
 'Energy Use (per ASK)_r2': np.float64(0.1944402081909089)}

In [96]:
_compute_polynomials_from_dataframe(
    df=df_with_dot,
    col_name_x='YOI',
    list_col_names_y=['L/D'],
    degree=2,
    plot=True
)

{'L/D': Polynomial([ 31.61287482,  -1.08079159, -20.23817236], domain=[1984., 2018.], window=[-1.,  1.], symbol='x'),
 'L/D_r2': np.float64(0.33499327897778297)}

In [97]:
_compute_polynomials_from_dataframe(
    df=df_with_dot,
    col_name_x='YOI',
    list_col_names_y=['TSFC (cruise)'],
    degree=2,
    plot=True
)

{'TSFC (cruise)': Polynomial([18.36325142, -3.93640552, -0.08288474], domain=[1958., 2018.], window=[-1.,  1.], symbol='x'),
 'TSFC (cruise)_r2': np.float64(0.6809264026394428)}

In [98]:
_compute_polynomials_from_dataframe(
    df=df_with_dot,
    col_name_x='YOI',
    list_col_names_y=['OEW/MTOW'],
    degree=2,
    plot=True
)

{'OEW/MTOW': Polynomial([ 0.53929443,  0.02823205, -0.03726449], domain=[1958., 2018.], window=[-1.,  1.], symbol='x'),
 'OEW/MTOW_r2': np.float64(0.09992500329622156)}

In [101]:
df_t2 = process_data_usdot_t2()

In [102]:
from aircraftdetective.data.constants import jeta1_energydensity

In [104]:
df_t2.columns

Index(['Year', 'Aircraft Designation (US DOT Schedule T2)',
       'Fuel/Available Seat Distance', 'Fuel/Revenue Seat Distance',
       'Fuel Flow', 'Energy Use (per ASK)', 'Airborne Efficiency', 'SLF'],
      dtype='object')

In [105]:
df_t2_filtered = df_t2[df_t2['Aircraft Designation (US DOT Schedule T2)'] == 'A220-300 BD-500-1A11']


In [106]:
df_t2_filtered['EI'] = df_t2_filtered['Fuel/Revenue Seat Distance'] * jeta1_energydensity

/var/folders/9l/8ctsrnn52fgbfv9t_tv0spfc0000gn/T/ipykernel_20071/863172944.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [111]:
df_t2_filtered['EI'].pint.to('MJ/km').min()

<Quantity(0.519409829, 'megajoule / kilometer')>

In [ ]:
df_t2_filtered = df_t2[df_t2['Aircraft De'] == 'JET']

In [112]:
df_t2

,Year,Aircraft Designation (US DOT Schedule T2),Fuel/Available Seat Distance,Fuel/Revenue Seat Distance,Fuel Flow,Energy Use (per ASK),Airborne Efficiency,SLF
0,1991,Boeing 757-200,0.01329377983359951,0.022279264205314737,1079.3313577391493,461294.160225903,0.8493508088032168,0.5966884593265996
1,1991,McDonnell Douglas DC-9-10,0.03305182276276791,0.06488049041833445,929.3542842646033,1146898.2498680465,0.7766995073891626,0.5094262165661414
2,1991,McDonnell Douglas DC-9-30,0.024863268467927784,0.047210231649932245,953.8429051869099,862755.4158370941,0.8001448975367419,0.5266499993537623
3,1991,McDonnell Douglas DC-9-40,0.025566983164445865,0.04696680102601686,950.8338278931751,887174.3158062715,0.7929411764705883,0.5443628819915424
4,1991,McDonnell Douglas DC-9-50,0.02415604775099102,0.04457317391750557,1110.591387226129,838214.8569593884,0.7821719169053138,0.5419413882371976
...,...,...,...,...,...,...,...,...
21146,2025,Airbus Industrie A320-100/200,0.012336433449789744,0.01587155588407961,974.4776785714286,428074.24070770416,0.797153024911032,0.777266799795232
21147,2025,Airbus Industrie A321/Lr,0.009550854362561301,0.01406695223141465,967.7333333333333,331414.6463808772,0.8477435681147195,0.6789569059054674
21148,2025,Airbus Industrie A321-200n,0.0072610377431749215,0.010733680523079494,791.6865191146882,251958.00968816978,0.8443764865783214,0.6764723179120418
21149,2025,Airbus Industrie A320-200n,0.009660906939498351,0.01309041806366368,781.1391349661282,335233.4708005928,0.8078299305409388,0.7380136289394035


In [113]:
df_t2.columns

Index(['Year', 'Aircraft Designation (US DOT Schedule T2)',
       'Fuel/Available Seat Distance', 'Fuel/Revenue Seat Distance',
       'Fuel Flow', 'Energy Use (per ASK)', 'Airborne Efficiency', 'SLF'],
      dtype='object')

In [114]:
df_engines_scaled

,Engine Identification,Fuel Flow (takeoff),Fuel Flow (climbout),Fuel Flow (approach),Fuel Flow (idle),B/P Ratio,Pressure Ratio,Rated Thrust,TSFC (takeoff),TSFC (cruise)
0,AE3007A,0.377,0.315,0.117,0.049,5.23,18.08,33.73,11.17699377408835,18.58610840687508
1,AE3007A1,0.3826,0.318,0.113,0.0461,4.77,17.97,34.91,10.959610426811803,18.363047179281903
2,AE3007A1 series,0.38,0.319,0.117,0.05,4.76,17.81,33.73,11.26593536910762,18.675850785971654
3,AE3007A1/1,0.3805,0.3163,0.1125,0.0459,4.77,17.9,34.74,10.95279217040875,18.3559654222498
4,AE3007A1/3,0.3589,0.2999,0.1077,0.0449,4.81,17.22,33.05,10.859304084720122,18.25834040527338
...,...,...,...,...,...,...,...,...,...,...
479,V2530-A5,1.331,1.077,0.377,0.138,4.54,32.1,133.4,9.97751124437781,17.28947569603809
480,V2530-A5 SelectOne™ Upgrade Package,1.325,1.078,0.387,0.145,4.6,32.0,133.0,9.962406015037594,17.272121935849103
481,V2531-E5,1.329,1.079,0.391,0.144,4.52,32.3,134.8,9.859050445103856,17.152696982705375
482,V2533-A5,1.426,1.1447,0.3901,0.1363,4.46,33.44,140.56,10.14513375071144,17.480338723125712


In [115]:
import plotly.express as px

In [117]:
fig = px.scatter(
    data_frame=df_engines_scaled, 
    x='B/P Ratio', 
    y='TSFC (cruise)'
)
fig.show()

AttributeError: Magnitude 'float' does not support tolist.

In [118]:
df_with_dot

,Manufacturer,Aircraft Designation,Aircraft Designation (Literature),Aircraft Designation (aircraft-database.com),Aircraft Designation (US DOT Schedule T2),Type,YOI,Engine Designation (ICAO),Engine Designation (aircraft-database.com),Design Range,...,Energy Use (per ASK),Airborne Efficiency,SLF,Engine Efficiency,Propulsive Efficiency,Thermal Efficiency,Aspect Ratio,L/D,OEW/MTOW,OEW/Exit Limit
0,Airbus,A220-100,NaN,A220-100,A200-100 BD-500-1A10,Narrow,2016,PW1519G,PW1519G,6700.0,...,1.1706056571081447,0.8278189758393693,0.7810071307439735,1.428333042570806e-06,0.8073414771820275,1.7691808025970729e-06,10.970703472840606,16.662689537570333,0.558161648177496,293.5
1,Airbus,A220-300,NaN,A220-300,A220-300 BD-500-1A11,Narrow,2016,PW1521G,PW1521G,6300.0,...,1.0921096365479033,0.8150626646059727,0.804697847717592,1.426404148253051e-06,0.7705664741746658,1.8511110930187252e-06,10.970703472840606,14.299599889133317,0.5229901269393512,264.85714285714283
2,Airbus,A300-600,A300-600,A300 B4-600,Airbus Industrie A300-600/R/CF/RCF,Wide,1984,CF6-80C2A5,CF6-80C2A5,6852.0,...,1.3645715712045796,0.8592071245393107,0.7205480386010795,1.1412162119658897e-06,0.7475432276078617,1.5266223675355625e-06,7.729726538461538,16.913520025725198,0.4800606060606061,219.41828254847644
3,Airbus,A300-600,A300-600,A300 B4-601,Airbus Industrie A300-600/R/CF/RCF,Wide,1986,PW4158,PW4158,6852.0,...,1.3645715712045796,0.8592071245393107,0.7205480386010795,1.1442508335980576e-06,0.751785921057727,1.5220434455438474e-06,7.729726538461538,16.868664359258453,0.4797030303030303,219.25484764542935
4,Airbus,A310-200,NaN,A310-200,Airbus Industrie A310-200C/F,Wide,1983,JT9D-7R4D*,JT9D-7R4D*,6500.0,...,1.685749511487712,0.8886528414373854,0.6141077924712185,1.1668621911251597e-06,0.7825336116487677,1.4911336379106154e-06,7.729726538461538,nan,nan,nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124,McDonnell Douglas,DC-10-10,DC10-10,DC-10,McDonnell Douglas DC-10-10,Wide,1971,NaN,NaN,nan,...,1.639364392629194,0.8697766384618486,0.6837898267755348,nan,nan,nan,nan,nan,nan,nan
125,McDonnell Douglas,DC-10-30,DC10-30,DC-10,McDonnell Douglas DC-10-30,Wide,1972,NaN,NaN,nan,...,1.7426594850018018,0.9200260927951432,0.7622711729058671,nan,nan,nan,nan,nan,nan,nan
126,McDonnell Douglas,DC-10-40,DC10-40,DC-10,McDonnell Douglas DC-10-40,Wide,1973,NaN,NaN,nan,...,1.6579148975075262,0.9006058031346184,0.7303027448385714,nan,nan,nan,nan,nan,nan,nan
127,Lockheed,L-1011-100,L1011-1/100/200,L-1011 TriStar,Lockheed L100-10 Hercules,Wide,1975,NaN,NaN,nan,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan


In [130]:
fig = px.scatter(
    data_frame=df_with_dot, 
    x='Year', 
    y='TSFC (cruise)_magnitude', # Reference the new magnitude column by name
    hover_data=['TSFC (cruise)_hover'], # Reference the new hover column by name
    labels={
        # Create a nice label for the y-axis
        'TSFC (cruise)_magnitude': f"TSFC (cruise) [{y_units:~P}]",
        'TSFC (cruise)_hover': 'TSFC (cruise)' # Clean up the hover label
    },
    title="Thrust-Specific Fuel Consumption Over Time"
)

# 4. SET AXIS LIMITS
# Define your desired limits. For the y-axis, use the correct units for clarity.
y_min_limit = 0
y_max_limit = 30
x_min_limit = 1955
x_max_limit = 2025

# Use the .update_xaxes() and .update_yaxes() methods.
# Provide the MAGNITUDES of the limits to the 'range' argument.
fig.update_yaxes(range=[y_min_limit, y_max_limit])
fig.update_xaxes(range=[x_min_limit, x_max_limit])


fig.show()